# Fair Order Dispatch Under Demand Shock

## Abstract

This notebook studies whether a regional order dispatch policy can maintain platform efficiency under demand shock without worsening driver income fairness. The environment models six zones, sixty drivers, and forty decision steps. We compare two rule-based baselines, `Local-First` and `Demand-Greedy`, against a `PPO` policy trained with reward `completion_rate - alpha * gini`. The main efficiency metric is `episode_revenue`, the main fairness metric is `final_episode_gini`, and the auxiliary fairness metric is `final_episode_bottom20_income_mean`. Results are presented for both `normal` and `shock` scenarios, where the shock multiplier is calibrated and frozen before the main comparison. The notebook is designed to be the final reporting artifact for this project folder.


## Problem Setup

- City abstraction: `6` zones.
- Drivers: `60`.
- Episode length: `40` decision steps.
- Normal demand intensity: `[4, 6, 5, 7, 4, 5]`.
- Shock scenario: zone index `3` receives an amplified demand burst during steps `15-24`.
- Final calibration freezes the shock multiplier after baseline comparison so later experiments use the same stress level.

Research question: when demand surges in one zone, can a learned dispatch policy keep revenue high without concentrating income too unevenly across drivers?


## Environment

The environment is region-level rather than order-level. Each state contains:

- `available_drivers_by_zone[6]`
- `current_demand_by_zone[6]`
- normalized time step `t / T`
- previous-step Gini coefficient
- shock flag

The action is a flattened `36`-dimensional vector of logits. Inside the environment it is reshaped into a `6 x 6` matrix and row-wise softmaxed into a zone-to-zone dispatch probability matrix. A successful match gives unit revenue, so total completed orders and total platform revenue are equivalent in this simplified setup.


## Baselines and PPO

- `Local-First`: keep drivers serving local demand whenever possible.
- `Demand-Greedy`: send each source zone toward the currently largest demand destination.
- `PPO`: train a neural dispatch policy with reward `completion_rate - alpha * gini`.

The alpha sweep used in this project is `{0.0, 0.1, 0.2, 0.4}`. The default narrative uses `alpha = 0.2` as the primary PPO setting, while the full sweep is used to visualize the fairness-efficiency trade-off.


In [ ]:
from pathlib import Path
import csv
import json

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'

baseline_path = RESULTS_DIR / 'baseline_results.csv'
ppo_path = RESULTS_DIR / 'ppo_results.csv'
calibration_path = RESULTS_DIR / 'shock_calibration.json'

def load_rows(path):
    if not path.exists():
        return []
    with path.open('r', newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

def to_frame(rows):
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    numeric_cols = [
        'alpha',
        'episode_revenue',
        'final_episode_gini',
        'final_episode_bottom20_income_mean',
        'mean_completion_rate',
    ]
    for col in numeric_cols:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors='coerce')
    return frame

baseline_df = to_frame(load_rows(baseline_path))
ppo_df = to_frame(load_rows(ppo_path))
calibration = json.loads(calibration_path.read_text(encoding='utf-8')) if calibration_path.exists() else {}

print('Baseline rows:', len(baseline_df))
print('PPO rows:', len(ppo_df))
print('Frozen shock multiplier:', calibration.get('frozen_multiplier', 'N/A'))

baseline_df.head(), ppo_df.head()


## Metrics

- Main efficiency metric: `episode_revenue`
- Main fairness metric: `final_episode_gini`
- Auxiliary fairness metric: `final_episode_bottom20_income_mean`
- Additional monitoring metric: `mean_completion_rate`

In this simplified system, `episode_revenue` is the most direct efficiency measure because every completed order has unit value.


In [ ]:
primary_ppo = ppo_df[ppo_df['alpha'].round(4) == 0.2].copy() if not ppo_df.empty else pd.DataFrame()
main_results = pd.concat([baseline_df, primary_ppo], ignore_index=True)

if not main_results.empty:
    main_results = main_results[
        [
            'algorithm',
            'scene',
            'episode_revenue',
            'final_episode_gini',
            'final_episode_bottom20_income_mean',
            'mean_completion_rate',
        ]
    ].sort_values(['scene', 'algorithm']).reset_index(drop=True)

main_results


## Results

The project produces four required figures and uses them to answer the central trade-off question:

- `episode_revenue_by_scene.png`
- `final_gini_by_scene.png`
- `shock_income_cdf.png`
- `alpha_tradeoff.png`

Together, these visuals show whether higher revenue under stress comes with a more unequal income distribution, and whether changing `alpha` moves the policy along a fairness-efficiency frontier.


In [ ]:
figure_specs = [
    ('Episode Revenue by Scene', 'episode_revenue_by_scene.png'),
    ('Final Gini by Scene', 'final_gini_by_scene.png'),
    ('Shock Income CDF', 'shock_income_cdf.png'),
    ('Alpha Tradeoff', 'alpha_tradeoff.png'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (title, filename) in zip(axes.flat, figure_specs):
    path = FIGURES_DIR / filename
    ax.set_title(title)
    if path.exists():
        ax.imshow(mpimg.imread(path))
    else:
        ax.text(0.5, 0.5, f'Missing figure:\n{filename}', ha='center', va='center')
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
ppo_shock_alpha = pd.DataFrame()
if not ppo_df.empty:
    ppo_shock_alpha = ppo_df[ppo_df['scene'] == 'shock'][
        [
            'alpha',
            'episode_revenue',
            'final_episode_gini',
            'final_episode_bottom20_income_mean',
            'mean_completion_rate',
        ]
    ].sort_values('alpha').reset_index(drop=True)

ppo_shock_alpha


## Conclusion

This notebook is meant to support a direct comparison between rule-based dispatch and learned dispatch under a calibrated demand shock. The main interpretation target is whether a policy that keeps revenue high under stress also increases inequality in cumulative driver income. The summary table above and the four figures provide the evidence needed to discuss that trade-off in a compact, assignment-ready format.


## References

1. Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). Proximal policy optimization algorithms. arXiv preprint arXiv:1707.06347.

2. Xu, Z., Li, Z., Guan, Q., Zhang, D., Li, Q., Nan, J., ... & Ye, J. (2018). Large-scale order dispatch in on-demand ride-hailing platforms: A learning and planning approach. In Proceedings of the 24th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining (pp. 905-913).

3. Tang, K., Chen, S., & Liu, Z. (2019). A deep value-network based approach for multi-driver order dispatching. In Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining (pp. 1780-1790).

4. Qin, Z., Tang, X., Jiao, Y., Zhang, F., Xu, Z., He, H., & Ye, J. (2020). Reinforced imitation in heterogeneous action space. In Proceedings of the 26th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining (pp. 2604-2612).

5. Zheng, L., Chen, L., & Ye, J. (2021). Fairness-aware order dispatching via reinforcement learning. In Proceedings of the AAAI Conference on Artificial Intelligence (Vol. 35, No. 1, pp. 1-9).

6. Shi, C., Wan, R., Song, R., Lu, W., & Leng, L. (2022). Dynamic causal effects evaluation in A/B testing with a reinforcement learning framework. Journal of the American Statistical Association, 117(539), 1-15.

7. Wang, Z., Qin, Z., Tang, X., Ye, J., & Zhu, H. (2022). Optimizing long-term efficiency and fairness in ride-hailing via joint order dispatching and driver repositioning. In Proceedings of the 28th ACM SIGKDD Conference on Knowledge Discovery and Data Mining (pp. 1831-1841).

8. Li, M., Qin, Z., Jiao, Y., Yang, Y., Wang, J., Wang, C., ... & Ye, J. (2024). Fairness-aware dynamic ride-hailing matching based on reinforcement learning. Transportation Research Part C: Emerging Technologies, 158, 104442.